In [0]:
df = spark.read.json('/Volumes/bigdata2/my_voloum/volume/orders_practice_2mb.json')


In [0]:
display(df)

In [0]:
df.columns

In [0]:
df.describe().show()

In [0]:
from pyspark.sql.functions import col
df2 = df.select (col("customer.city").alias("customer_city"), 
                 col("customer.name").alias("customer_name"), 
                 "order_date",
                 "price",
                 "qty",
                 "status",
                 "payment")
df2.show(2)                 

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import year, to_date , month

df=df.withColumn("order_data" , to_date("order_date"))\
    .withColumn("order_year" , year("order_date"))\
        .withColumn("order_month" , month("order_date"))

df.show(2)        

In [0]:
from pyspark.sql.functions import date_format

df=df.withColumn("month_name" , date_format("order_date" , "MMMM"))\
    .withColumn("weekday_name" , date_format("order_date" , "EEEE"))

display(df)



In [0]:
df.where( col("order_month").isin("1","2","3","4","5","6","7","8","9","10","11","12"))

In [0]:
from pyspark.sql.functions import when , dayofweek

df=df.withColumn("day_type",
                 when(dayofweek(col("order_date")).isin(1,7),"weekend")
                 .otherwise("weekday"))
 
display(df) 

In [0]:
from pyspark.sql.functions import col

df = df.withColumn(
    "order_date_dt",
    col("order_date").cast("date")
)

df.printSchema()

In [0]:
df = df.drop("order_date_dt")

In [0]:
from pyspark.sql.functions import quarter

df = df.withColumn("order_quarter", quarter(col("order_date")))

In [0]:
from pyspark.sql.functions import date_format

df = df.withColumn("month_name", date_format(col("order_date"), "MMMM")) \
       .withColumn("weekday_name", date_format(col("order_date"), "EEEE"))

In [0]:
from pyspark.sql.functions import when, dayofweek

df = df.withColumn(
    "day_type",
    when(dayofweek(col("order_date")).isin(1, 7), "weekend")
    .otherwise("weekday")
)

In [0]:
display(df.select(
    "order_date",
    "order_year",
    "order_month",
    "order_quarter",
    "month_name",
    "weekday_name",
    "day_type"
))


In [0]:
from pyspark.sql.functions import sum

df.groupBy("customer.city") \
  .agg(sum("price").alias("total_sales")) \
  .show()

In [0]:
from pyspark.sql.functions import sum, avg

df.groupBy("order_year", "order_month") \
  .agg(
      sum("qty").alias("total_qty"),
      avg("price").alias("avg_price")
  ).show()

In [0]:
from pyspark.sql.functions import count

df.groupBy("status") \
  .agg(count("*").alias("order_count")) \
  .show()

In [0]:
from pyspark.sql.functions import sum, col

total_revenue_df = df.select(
    (col("price").cast("double") * col("qty").cast("int")).alias("revenue")
).agg(
    sum("revenue").alias("total_revenue")
)

total_revenue_df.show()

In [0]:
from pyspark.sql.functions import countDistinct

unique_orders_df = df.agg(
    countDistinct("order_id").alias("unique_order_count")
)

unique_orders_df.show()

In [0]:
from pyspark.sql.functions import year, countDistinct, col

df.filter(year(col("order_date")).isin(2025, 2026)) \
  .groupBy(year(col("order_date")).alias("order_year")) \
  .agg(countDistinct("order_id").alias("no_of_orders")) \
  .orderBy("order_year") \
  .show()

In [0]:
from pyspark.sql.functions import sum, col

df.groupBy("customer.name") \
  .agg(sum(col("price").cast("double") * col("qty").cast("int")).alias("total_amount_spend")) \
  .orderBy(col("total_amount_spend").desc()).show()

In [0]:
from pyspark.sql.functions import sum, avg, countDistinct , col 

df = df.withColumn(
    "amount",
    col("price").cast("double") * col("qty").cast("int")
)

city_summary_df = df.groupBy("customer.city") \
    .agg(
        countDistinct("order_id").alias("no_of_orders"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("avg_amount_per_order")
    ) \
    .orderBy("customer.city")

city_summary_df.show()

In [0]:
from pyspark.sql.functions import col, sum

missing_df = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_df.show()

In [0]:
from pyspark.sql.functions import col, sum

missing_counts = [
    (c, df.filter(col(c).isNull()).count())
    for c in df.columns
]

spark.createDataFrame(
    missing_counts, ["column_name", "missing_count"]
).show()

In [0]:
by using cast function 1 and 7 are weekend

make 3 quter of the function 

group by and window function and aggregate